In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

In [3]:
from celavi.data_manager import PVTechUnitLocations, PVTechUnitChars
from celavi.compute_locations import ComputeLocations

In [4]:
start_year = 2010

In [5]:
chars = Path('C:/Users/rhanes/GitHub/celavi-data/inputs_to_preprocessing/glasspermodule_pvice.csv')
locs = Path('C:/Users/rhanes/GitHub/celavi-data/inputs_to_preprocessing/uspvdb_v1_0_20231108.csv')
fac_type = Path('C:/Users/rhanes/GitHub/celavi-data/inputs/facility_type.csv')

In [6]:
modules = PVTechUnitChars(fpath=chars, backfill=True)

validating PVTechUnitChars
validated PVTechUnitChars
no missing data values in celavi.data_manager.PVTechUnitChars.MWdc_per_m2
no missing data values in celavi.data_manager.PVTechUnitChars.MWdc_per_module
no missing data values in celavi.data_manager.PVTechUnitChars.glass_metrictonne_per_module


In [14]:
facility_type_lookup = pd.read_csv(fac_type, header=None)

In [18]:
# Process data for solar power plants - from USPVDB
pv_locs_raw = PVTechUnitLocations(fpath = locs, backfill = True)

# select only those plants with eia_ids 
pv_locs = pv_locs_raw[(pv_locs_raw['eia_id'] != '-1') & (pv_locs_raw['p_year'] != '-1')]

# reformat data for later use
pv_locs = pv_locs.rename(
    columns={
        "p_state": "region_id_2",
        'p_county': 'region_id_3',
        "xlong": "long",
        "ylat": "lat",
        "eia_id": "facility_id",
        "p_year": "year"
        },
        )

# exclude Hawaii, Guam, Puerto Rico, and Alaska (only have road network data for the contiguous United States)
pv_locs.drop(
    index = pv_locs[pv_locs.region_id_2.isin(['HI','GU','PR','AK'])].index,
    inplace = True
)

# exclude Nantucket since transport routing doesn't currently include ferries
pv_locs.drop(
    index = pv_locs[pv_locs.region_id_3 == 'Nantucket'].index,
    inplace = True
)

# exclude all PV subtypes other than c-Si
# that's the only one we have glass information on
pv_locs.drop(
    index = pv_locs[pv_locs.p_tech_sec != 'c-si'].index,
    inplace = True
)

# then the tech_sec column is no longer needed
pv_locs.drop(
    columns = 'p_tech_sec',
    inplace = True
)

# also drop data from before the simulation start year
pv_locs.drop(
    index = pv_locs[pv_locs.year < start_year].index,
    inplace = True
)

# Filter down the dataset to generate the number_of_technology_units
# file
# Store this dataframe into self for use in capacity projection
# calculations and creation of the number_of_technology_units file
capacity_data = pv_locs[
    ['facility_id', 'p_name', 'year', 'p_cap_dc']
    ].drop_duplicates().dropna()

# Aggregate to STATE level by SUMMING capacity and AVERAGING
# location. State = region_id_2
# (this is the plant location for each facility_id)
fac_ids_state = pv_locs[['facility_id','region_id_2']].drop_duplicates(
    subset='region_id_2', keep='last'
    )
pv_locs_state = pv_locs.groupby(
    ['region_id_2','year']
    ).agg(
        {'lat': np.mean, 'long': np.mean, 'p_cap_dc': 'sum'}
        ).reset_index(
        ).merge(
            fac_ids_state, on='region_id_2', how='outer'
            )

power_plant_type_lookup = facility_type_lookup[facility_type_lookup[0].str.contains('power plant')].values[0][0]
if power_plant_type_lookup:
    pv_locs_state["facility_type"] = power_plant_type_lookup
else:
    warnings.warn('Power plant facility type missing from facility_type lookup table.')

pv_locs_state["region_id_1"] = 'USA'
pv_locs_state["region_id_4"] = ''

validating PVTechUnitLocations
validated PVTechUnitLocations
no missing data values in celavi.data_manager.PVTechUnitLocations.eia_id
no missing data values in celavi.data_manager.PVTechUnitLocations.p_year


In [19]:
pv_locs_state

,region_id_2,year,lat,long,p_cap_dc,facility_id,facility_type,region_id_1,region_id_4
0,AL,2015.0,32.304798,-84.990997,41.5,59862,pv power plant,USA,
1,AL,2016.0,34.833801,-87.838303,100.2,59862,pv power plant,USA,
2,AL,2017.0,33.741549,-86.010700,129.4,59862,pv power plant,USA,
3,AR,2016.0,36.164200,-94.082397,1.7,62683,pv power plant,USA,
4,AR,2017.0,35.052750,-93.046848,8.0,62683,pv power plant,USA,
...,...,...,...,...,...,...,...,...,...
301,WI,2017.0,44.582627,-91.120992,20.5,60893,pv power plant,USA,
302,WI,2018.0,45.210602,-91.433296,3.7,60893,pv power plant,USA,
303,WI,2020.0,43.513866,-88.800199,230.7,60893,pv power plant,USA,
304,WI,2021.0,42.997700,-89.453400,24.2,60893,pv power plant,USA,


In [ ]:
pv_locs_state = pv_locs.groupby(
    ['region_id_2','year']
).agg(
    {'lat': np.mean, 'long': np.mean, 'p_cap_dc': 'sum'}
).reset_index(
).merge(
    fac_ids_state, on='region_id_2', how='outer'
)
pv_locs_state.head()